# Tracer + ToyAgent — audit your own toy agent

This notebook walks you through **defining a toy agent as a sequence of Python steps** and **auditing it with Tracer's LLM judge**. By the end you will have:

1. a tiny agent you wrote,
2. a step-level blame report from Tracer,
3. a before/after demo you can drop straight into your Week 5 slides.

The pattern is **backbone-agnostic**: your step functions can call OpenAI, Anthropic, a local model, or nothing at all. Tracer only sees each step's inputs and outputs.

## 0. Setup

Make sure:
- `pandas` and `openai` are installed (`pip install -r requirements.txt`),
- the Tracer repo at `../../materials/Tracer/` exists,
- your `OPENAI_API_KEY` is set in the environment (or will be set below).

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY (and GROQ_API_KEY) from .env

# 1) Make Tracer importable. It's located in the home directory root.
TRACER_DIR = Path.home() / "Tracer"
assert TRACER_DIR.exists(), f"Tracer not found at {TRACER_DIR}"
if str(TRACER_DIR) not in sys.path:
    sys.path.insert(0, str(TRACER_DIR))

# 2) Make our ToyAgent helper importable (this notebook's directory).
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from toy_agent import ToyAgent             # our ~120-line helper
from parser import parse_source             # Tracer modules
from executor import TracingExecutor
from judge import LLMJudge
from reporter import Reporter

print("Tracer dir:", TRACER_DIR)
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## 1. Define your agent's goal

The **goal** is the one-sentence description Tracer's judge will evaluate each step against. Be specific about what `CORRECT` means.

In [2]:
agent = ToyAgent(
    goal="Find the top 3 customers by total spending and return their names, highest spender first.",
)

## 2. Register steps

Two pieces of setup first:

- **`agent.imports`** — import lines prepended to the generated script.
- **`agent.setup`** — a block of Python code that creates a variable named `state`. This is whatever the *first* step will receive as input. Any tables, constants, or LLM clients your pipeline needs should be reachable from `state`.

**Convention:** each step takes the previous step's return value and returns whatever the next step will consume. By convention `state` is a dict the steps mutate, but any threaded value works. The last step can return the final answer directly.

Why the `state` dict instead of closures over module-level tables? Tracer wraps each step function at *definition time* — once wrapped, the function can no longer see variables declared in the enclosing module. Threading `state` through the arguments sidesteps that. Give each step a docstring that says what it reads from `state` and what it writes back — the judge uses it as context.

In [3]:
agent.imports = [
    "import pandas as pd",
]

# `setup` must create a variable named `state` --- the initial input for
# the first step. We bundle both tables into it.
agent.setup = """
    state = {
        'customers': pd.DataFrame({
            'customer_id': [1, 2, 3, 4, 5],
            'name':        ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
        }),
        'orders': pd.DataFrame({
            'order_id':     [101, 102, 103, 104, 105, 106],
            'customer_id':  [  1,   2,   1,   3,   2,   4],
            'total_amount': [150.0, 200.0, 300.0, 175.5, 250.0, 400.0],
        }),
    }
"""

In [4]:
@agent.add_step
def build_totals(state):
    """Aggregate total spending per customer.

    Adds state['totals']: a DataFrame with ['customer_id', 'total_amount'].
    """
    state['totals'] = (
        state['orders']
        .groupby('customer_id')['total_amount']
        .sum()
        .reset_index()
    )
    return state


@agent.add_step
def pick_top_n(state):
    """Pick the top 3 customers by total spending.

    Reads state['totals']; adds state['top'] sorted HIGHEST to LOWEST.
    """
    # BUG (intentional): ascending=True returns the three LOWEST spenders.
    state['top'] = state['totals'].sort_values('total_amount', ascending=True).head(3)
    return state


@agent.add_step
def format_result(state):
    """Join customer names and return the top-3 names as a list (final output)."""
    merged = state['customers'].merge(state['top'], on='customer_id')
    return merged['name'].tolist()


print("Registered steps:", [s.name for s in agent.steps])

Registered steps: ['build_totals', 'pick_top_n', 'format_result']


## 3. Sanity-run the agent (no Tracer yet)

Before auditing, just run the agent end-to-end. The point is to confirm it **finishes without crashing** — that is exactly the kind of silent failure error localization exists for.

In [5]:
print("agent.run_local() =>", agent.run_local())
# Expected (with the intentional bug): ['Alice', 'Carol', 'David'] --- the LOWEST spenders.

agent.run_local() => ['Alice', 'Carol', 'David']


## 4. See what Tracer will see

`agent.to_script()` stitches imports + init_code + step sources + a small runner into one Python source string. That string is what Tracer's parser and executor will consume.

In [6]:
print(agent.to_script())

# Auto-generated by ToyAgent for Tracer.
# Goal: Find the top 3 customers by total spending and return their names, highest spender first.

import pandas as pd

# --- setup: build the initial `state` ---
state = {
    'customers': pd.DataFrame({
        'customer_id': [1, 2, 3, 4, 5],
        'name':        ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    }),
    'orders': pd.DataFrame({
        'order_id':     [101, 102, 103, 104, 105, 106],
        'customer_id':  [  1,   2,   1,   3,   2,   4],
        'total_amount': [150.0, 200.0, 300.0, 175.5, 250.0, 400.0],
    }),
}

def build_totals(state):
    """Aggregate total spending per customer.

    Adds state['totals']: a DataFrame with ['customer_id', 'total_amount'].
    """
    state['totals'] = (
        state['orders']
        .groupby('customer_id')['total_amount']
        .sum()
        .reset_index()
    )
    return state

def pick_top_n(state):
    """Pick the top 3 customers by total spending.

    Reads state['totals']; adds

## 5. Audit with Tracer

One call: parse the script, execute step-by-step with the judge attached, report findings. `continue_on_error=True` makes Tracer keep going past errors so we collect **every** bug in one pass.

In [7]:
#%pip install openai

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from .env

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in your .env file before running the audit cell.")

result = agent.audit_with_tracer(api_key=api_key)

## 6. Read the findings

Each entry in `result.errors` has a line number, error type, and message. The logical errors are the interesting ones — they are the silent bugs that `run_local` did not surface.

In [9]:
print(f"Tracer found {len(result.errors)} issue(s):\n")
for err in result.errors:
    print(f"  line {err.lineno:>3}  {err.error_type:<12}  {err.error_message}")

Tracer found 0 issue(s):



## 7. Fix the bug and re-audit — the "before/after" slide

We rebuild the agent with `ascending=False`. Everything else is identical. The `pick_top_n` verdict should flip from `INCORRECT` to `CORRECT`, and your `result.errors` list should shrink.

In [10]:
fixed_agent = ToyAgent(goal=agent.goal)
fixed_agent.imports = agent.imports
fixed_agent.setup = agent.setup


@fixed_agent.add_step
def build_totals(state):
    """Aggregate total spending per customer."""
    state['totals'] = (
        state['orders']
        .groupby('customer_id')['total_amount']
        .sum()
        .reset_index()
    )
    return state


@fixed_agent.add_step
def pick_top_n(state):
    """Pick the top 3 customers by total spending, HIGHEST first."""
    state['top'] = state['totals'].sort_values('total_amount', ascending=False).head(3)  # FIXED
    return state


@fixed_agent.add_step
def format_result(state):
    """Return the top-3 customer names."""
    merged = state['customers'].merge(state['top'], on='customer_id')
    return merged['name'].tolist()


print("fixed_agent.run_local() =>", fixed_agent.run_local())
print()
fixed_result = fixed_agent.audit_with_tracer(api_key=api_key)
print()
print(f"Before: {len(result.errors)} issue(s)")
print(f"After:  {len(fixed_result.errors)} issue(s)")

fixed_agent.run_local() => ['Alice', 'Bob', 'David']

final: ['Alice', 'Bob', 'David']

=== Execution Result ===

Status: SUCCESS
Reason: completed
Steps executed: 9
Function calls: 3

Function Call Summary:
  [??] build_totals() -> {'customers':    customer_id   name
0            1
  [??] pick_top_n() -> {'customers':    customer_id   name
0            1
  [??] format_result() -> ['Alice', 'Bob', 'David']

Final Variables:
  build_totals = <function build_totals>
  pick_top_n = <function pick_top_n>
  format_result = <function format_result>
  pd = <module 'pandas' from '/Users/katyaogai/Documents/dbt/.venv/lib/python3.13/site-packages/pandas/__in
  state = ['Alice', 'Bob', 'David']


Before: 0 issue(s)
After:  0 issue(s)


## 8. Bring your own LLM backbone

Your step functions can call **anything**. Tracer only sees the step's inputs and outputs. Below are three equivalent shapes for a step that asks an LLM to draft a plan — pick whichever matches your project.

In [11]:
# Option A --- OpenAI (as an agent step that reads state["question"] and
# adds state["plan"]):
#
# from openai import OpenAI
# _client = OpenAI()
#
# @agent.add_step
# def draft_plan(state):
#     """Ask the planner LLM for a short plan. Adds state['plan']."""
#     resp = _client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": f"Plan: {state['question']}"}],
#     )
#     state["plan"] = resp.choices[0].message.content
#     return state

# Option B --- Anthropic
#
# from anthropic import Anthropic
# _client = Anthropic()
#
# @agent.add_step
# def draft_plan(state):
#     """Ask the planner LLM for a short plan. Adds state['plan']."""
#     msg = _client.messages.create(
#         model="claude-haiku-4-5",
#         max_tokens=256,
#         messages=[{"role": "user", "content": f"Plan: {state['question']}"}],
#     )
#     state["plan"] = msg.content[0].text
#     return state

# Option C --- mock (no LLM; useful for offline dev)
#
# @agent.add_step
# def draft_plan(state):
#     """Return a stubbed plan so we can wire the rest first."""
#     state["plan"] = "1) filter rows  2) sort  3) take top 3"
#     return state

print("Uncomment whichever backbone you want. Tracer does not care.")

Uncomment whichever backbone you want. Tracer does not care.


## Next steps — for your Week 5 demo

1. **Swap in your own agent**: replace the three example steps with whatever your team's agent actually does. Keep each step small and give it a tight docstring.
2. **Keep the goal specific**: "return top 3 customers, highest first" is auditable; "be helpful" is not.
3. **Screenshot the two `result.errors` lists** (before/after) — that is your before/after slide.
4. If you want more than logical-error detection, point Tracer at your agent's emitted scripts directly via the CLI: `python tracer.py mine.py --goal "…"` (see `../../materials/Tracer/README.md`).

That's it. The whole auditing layer is roughly 20 lines of glue plus Tracer.